In [1]:
from manim import *
import numpy as np
config.media_width = "75%"
config.verbosity = "WARNING"

config.background_color = WHITE

# Those are objects which are WHITE by default
# Define the wanted color for each one
white_objects ={
    Angle: ('dot_color', BLACK),
    AnnotationDot: ('stroke_color', BLACK),
    AnnularSector: ('color', BLACK),
    Annulus: ('color', BLACK),
    Arrow: ('color', BLACK),
    Arrow3D: ('color', BLACK),
    ArrowVectorField: ('color', BLACK),
    Code: ('background_stroke_color', BLACK),
    CubicBezier: ('color', BLACK),
    DashedVMobject: ('color', BLACK),
    Dot: ('color', BLACK),
    Dot3D: ('color', BLACK),
    Line: ('color', BLACK),
    Line3D: ('color', BLACK),
    MarkupText: ('color', BLACK),
    Polygon: ('color', BLACK),
    Rectangle: ('color', BLACK),
    SingleStringMathTex: ('color', BLACK),
    StreamLines: ('color', BLACK),
    Text: ('color', BLACK),
    TracedPath: ('stroke_color', BLACK),
    VectorField: ('color', BLACK),
    Arc: ('stroke_color', BLACK),
    Brace: ('color', BLACK)
}

for obj, (attr, color) in white_objects.items():
    obj.set_default(**{attr: color})

# Other configurations
Table.set_default(line_config=
    {"stroke_width": 1, "stroke_opacity": 0.5, "color": BLACK})

Code.set_default(style="pastie")
YELLOW="#ebe534"

In [24]:
%%manim -qh BrownianPaths

class BrownianPaths(Scene):
    def construct(self):
        dot = Dot(color=RED, radius=1)
        path = VGroup()  # Group to store the path
        self.add(dot, path)
        
        num_steps = 300  # Number of steps in the motion
        step_size = 0.15  # Maximum step size
        
        last_position = dot.get_center()
        
        for _ in range(num_steps):
            dx, dy = np.random.uniform(-step_size, step_size, size=2)  # Random step
            new_position = last_position + np.array([dx, dy, 0])
            
            line = Line(last_position, new_position, color=BLUE)
            path.add(line)
            
            self.play(dot.animate.move_to(new_position), run_time=0.1)
            last_position = new_position
        
        self.wait()


Manim Community v0.18.1

In [31]:
%%manim -qh WienerDef

class WienerDef(Scene):
    def construct(self):
        title = Tex(r"\textbf{Definition of a Wiener process}")
        def1 = Tex(r"A stochastic process $W_t$ is a Wiener process if").shift(UP*1.5)
        def2 = Tex(r"""
            \begin{enumerate}
            \item $\mathbb{P}(W_0 = 0) = 1$
            \item $W_{t+u} - W_t$ is independent of $W_s$ for $s < t$
            \item $W_{t+u} - W_t \sim \mathcal{N}(0,u)$
            \item $W_t$ has almost surely continuous sample paths
            \end{enumerate}
        """).next_to(def1, DOWN)

        title.move_to(UP*3)
        self.play(FadeIn(title))
        self.play(Write(def1))
        self.play(Write(def2))
        self.wait(3)
        self.play(FadeOut(title, def1, def2))

Manim Community v0.18.1

In [2]:
%%manim -qh TotalVariationScene

class TotalVariationScene(Scene):
    def construct(self):
        # Write the title at the top
        title = Tex("Total variation", font_size=48)
        title.to_edge(UP)
        self.play(Write(title))
        self.wait(1)

        # Create coordinate axes
        axes = Axes(
            x_range=[-2, 2, 0.5],
            y_range=[-2, 2, 0.5],
            x_length=6,
            y_length=4,
            tips=False,
        )
        axes.shift(DOWN*1.5)
        axes_labels = axes.get_axis_labels(x_label="x", y_label="f(x)")

        # Define a continuous function, e.g. f(x) = x^3 - x
        def func(x):
            return x**3 - x

        graph = axes.plot(func, color=BLUE, x_range=[-1.5, 1.5])
        self.play(Create(axes), Write(axes_labels), Create(graph))
        self.wait(1)

        # Create two dots:
        # Dot that travels along the graph
        dot_graph = Dot(color=RED).move_to(axes.c2p(-1.5, func(-1.5)))
        # Dot that remains left of the graph but moves vertically with f(x)
        dot_left = Dot(color=GREEN).move_to(axes.c2p(-1.5, func(-1.5)) + LEFT*2)
        self.play(FadeIn(dot_graph), FadeIn(dot_left))
        self.wait(0.5)

        # Use a ValueTracker to animate the movement along the graph
        x_tracker = ValueTracker(-1.5)
        dot_graph.add_updater(
            lambda m: m.move_to(axes.c2p(x_tracker.get_value(), func(x_tracker.get_value())))
        )
        dot_left.add_updater(
            lambda m: m.move_to(axes.c2p(-1.5, func(x_tracker.get_value())) + LEFT*2)
        )

        self.play(x_tracker.animate.set_value(1.5), run_time=4, rate_func=linear)
        self.wait(1)

        # Remove updaters after animation
        dot_graph.clear_updaters()
        dot_left.clear_updaters()

        # Illustrate a partition on the function
        # Here we choose a simple partition: x = -1.5, 0, 1.5
        partition_points = [-1.5, -1/np.sqrt(3), 1/np.sqrt(3), 1.5]
        dots_partition = VGroup(*[
            Dot(axes.c2p(x, func(x)), color=YELLOW) for x in partition_points
        ])
        self.play(FadeIn(dots_partition))
        self.wait(0.5)
        # Draw dashed vertical lines from partition points down to the x-axis
        dashed_lines = VGroup(*[
            DashedLine(axes.c2p(partition_points[i], func(partition_points[i])), axes.c2p(partition_points[i], func(partition_points[i+1])), color=GRAY) 
            for i in range(len(partition_points)-1)
        ])
        self.play(Create(dashed_lines))
        self.wait(2)
        self.play(dashed_lines.animate.arrange(UP, buff=0).rotate(PI/2).shift(UP*2))
        self.wait(2)

        # Write the definition of total variation above the curve
        total_variation_text = MathTex(
            r"TV(f, [a,b]) = \sup_{P} \sum_{i=0}^{n-1} \left| f(t_{i+1}) - f(t_i) \right|"
        )
        total_variation_text.next_to(graph, UP, buff=1)
        self.play(FadeOut(dashed_lines))
        self.play(Write(total_variation_text))
        self.wait(2)

        diff = Tex(r"If $f$ is differentiable, then $TV(f, [a,b]) = \int_a^b |f'(x)|\,dx$.")
        self.play(FadeOut(dots_partition, axes, axes_labels, dot_left, graph, dot_graph))
        self.play(Write(diff))
        self.wait(2)

        brown = MathTex(r"TV(W_t, [a,b]) = \infty", font_size=80).shift(DOWN*2)
        self.play(FadeIn(brown))
        self.wait()
        self.play(FadeOut(title, total_variation_text, diff, brown))

Manim Community v0.18.1

In [3]:
%%manim -qh QuadraticVariation

class QuadraticVariation(Scene):
    def construct(self):
        title = Tex(r"Quadratic variation", font_size=80).to_edge(UP)
        self.play(Write(title))
        self.wait()
        
        def1 = MathTex(r"[X]_{t}=\lim _{\Vert P\Vert \rightarrow 0}\sum _{k=1}^{n}(X_{t_{k}}-X_{t_{k-1}})^{2}").next_to(title, DOWN)
        self.play(FadeIn(def1))
        self.wait()

        brown = Tex(r"If $W_t$ is a Wiener process, then $[W]_t = t$.", font_size=65).shift(DOWN*0.5)
        self.play(FadeIn(brown))

        self.wait(2)
        self.play(FadeOut(title, def1, brown))

Manim Community v0.18.1